##Bronze to Silver: Data transformation and cleansing

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DoubleType, TimestampType, DateType
import pyspark.sql.functions as F

catalog_name = 'ecommerce'

##Brands

In [0]:
bronze_brands = spark.table(f'{catalog_name}.bronze.brz_brands')
bronze_brands.show(10, truncate=False)

In [0]:
silver_brands = bronze_brands.withColumn('brand_name', F.trim(F.col('brand_name')))
silver_brands = silver_brands.withColumn('brand_code', F.regexp_replace(F.col('brand_code'), r'[^A-Za-z0-9]', ''))
silver_brands.show(10)

In [0]:
silver_brands.select('category_code').distinct().show()

In [0]:
# Anomalies dictionary
anomalies = {
    "GROCERY": "GRCY",
    "BOOKS": "BKS",
    "TOYS": "TOY"
}

# PySpark replace is easy
silver_brands = silver_brands.replace(to_replace=anomalies, subset=["category_code"])

# ✅ Show results
silver_brands.select("category_code").distinct().show()

In [0]:
silver_brands.display()

In [0]:
silver_brands.write.format('delta') \
    .mode('overwrite') \
    .option('mergeSchema', 'true') \
    .saveAsTable(f'{catalog_name}.silver.slv_brands')

##Category

In [0]:
bronze_category = spark.table(f'{catalog_name}.bronze.brz_category')
bronze_category.display()

In [0]:
silver_category = bronze_category.dropDuplicates(['category_code'])
silver_category = silver_category.withColumn('category_code', F.upper(F.col('category_code')))
silver_category.display()

In [0]:
silver_category.write.format('delta').mode('overwrite').option('mergeSchema', 'true').saveAsTable(f'{catalog_name}.silver.slv_category')

##Products

In [0]:
bronze_products = spark.table(f'{catalog_name}.bronze.brz_products')
display(bronze_products)

In [0]:
silver_products = bronze_products.withColumn('weight_grams', F.regexp_replace(F.col('weight_grams'), 'g', '').cast('int'))
silver_products = silver_products.withColumn('length_cm', F.regexp_replace(F.col('length_cm'), ',', '.').cast('double'))
silver_products = silver_products.withColumn('category_code', F.upper(F.col('category_code'))).withColumn('brand_code', F.upper(F.col('brand_code')))
silver_products.display()

In [0]:
silver_products = silver_products.withColumn('material', F.when(F.col('material')=='Coton', 'Cotton').when(F.col('material')=='Alumium', 'Aluminum').when(F.col('material')=='Ruber', 'Rubber').otherwise(F.col('material')))
silver_products.display()

In [0]:
silver_products.select('material').distinct().display()

In [0]:
silver_products.select('rating_count').filter(F.col('rating_count')<0).display()

In [0]:
silver_products=silver_products.withColumn('rating_count', F.when(F.col('rating_count').isNotNull(), F.abs(F.col('rating_count'))).otherwise(F.lit(0)))
silver_products.display()

In [0]:
silver_products.write.format('delta').mode('overwrite').option('mergeSchema', 'true').saveAsTable(f'{catalog_name}.silver.slv_products')

##Customers

In [0]:
bronze_customers = spark.table(f'{catalog_name}.bronze.brz_customers')
bronze_customers.display()

In [0]:
silver_customers = bronze_customers.dropna(subset=['customer_id'])
silver_customers = silver_customers.withColumn('phone', F.col('phone').cast('bigint'))
silver_customers = silver_customers.withColumn('phone', F.col('phone').cast('string'))
silver_customers = silver_customers.fillna('Not available', subset=['phone'])
silver_customers.display()

In [0]:
silver_customers.write.format('delta').mode('overwrite').option('mergeSchema', 'true').saveAsTable(f'{catalog_name}.silver.slv_customers')

## Calendar/Dates

In [0]:
bronze_calendar = spark.table(f'{catalog_name}.bronze.brz_calendar')
bronze_calendar.display()

In [0]:
silver_calendar = bronze_calendar.withColumn('day_name', F.initcap(F.col('day_name')))
silver_calendar = silver_calendar.withColumn('week_of_year', F.abs('week_of_year'))
silver_calendar = silver_calendar.dropDuplicates(subset=['date'])
silver_calendar = silver_calendar.withColumn('quarter', F.concat(F.lit('Q'), F.col('quarter'), F.lit('-'), F.col('year')))
silver_calendar = silver_calendar.withColumn('week_of_year', F.concat(F.lit('Week'), F.col('week_of_year'), F.lit('-'), F.col('year')))
silver_calendar.display()

In [0]:
silver_calendar.write.format('delta').mode('overwrite').option('mergeSchema', 'true').saveAsTable(f'{catalog_name}.silver.slv_calendar')